##  Deceleration Rate to Avoid Crash (DRAC) Analysis

In [1]:
import time

In [4]:
# imports
import pandas as pd
import os
from datetime import datetime, timedelta
import numpy as np

START_DATE = "2025-06-02"
END_DATE = "2025-06-02"

DATA_DIR = "C:/Users/suggu/IITM/AGC/flow-analytics/Data/2025-06-02-data/2nd_June_2025"

def load_data(data_dir, start_date, end_date):
    
    # List to collect dataframes
    dfs = []
    
    # Loop over all folders within date range
    for folder in sorted(os.listdir(data_dir)):
        folder_path = os.path.join(data_dir, folder)
        
        # Skip non-folders
        if not os.path.isdir(folder_path):
            continue
        
        if folder.startswith(START_DATE) or folder.startswith(END_DATE):
            # Load all parquet files inside the folder
            dfs.append(pd.read_parquet(folder_path))
    
    # Combine all into single dataframe
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        return df
    else:
        print("No data found for given date range.")
        return pd.DataFrame()

# Usage
df = load_data(DATA_DIR, START_DATE, END_DATE)
print(f"Loaded {len(df)} records from {START_DATE} to {END_DATE}")

Loaded 48283533 records from 2025-06-02 to 2025-06-02


In [5]:
df.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-02 00:00:00.041107893,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.236328,-0.0,0.017080
1,2025-06-02 00:00:00.140780926,8748438,4,61.93750,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.244141,0.0,0.018615
2,2025-06-02 00:00:00.249500036,8748438,4,61.93750,-58.50000,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.242188,-0.0,0.025484
3,2025-06-02 00:00:00.338927984,8748438,4,61.93750,-58.46875,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.238281,-0.0,0.006648
4,2025-06-02 00:00:00.436949015,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.236328,-0.0,0.027398


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48283533 entries, 0 to 48283532
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


In [7]:
df['timestamp']

0          2025-06-02 00:00:00.041107893
1          2025-06-02 00:00:00.140780926
2          2025-06-02 00:00:00.249500036
3          2025-06-02 00:00:00.338927984
4          2025-06-02 00:00:00.436949015
                        ...             
48283528   2025-06-02 23:59:59.550574064
48283529   2025-06-02 23:59:59.654936075
48283530   2025-06-02 23:59:59.747447968
48283531   2025-06-02 23:59:59.841057062
48283532   2025-06-02 23:59:59.942635059
Name: timestamp, Length: 48283533, dtype: datetime64[ns]

In [8]:
df['timestamp'].duplicated().sum()

np.int64(47420264)

In [9]:
df.reset_index(drop=True, inplace=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48283533 entries, 0 to 48283532
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


### 1. Lifetime Filtering

In [12]:
# Global minimum detection count required per label
global_lifespan_thresholds = {
        1: 30,
        2: 80,
        3: 60,
        4: 90,
        5: 30,
        6: 100,
        7: 100,   
        8: 180
    }

# Compute lifespan as detection count in full dataset
lifespan = (
    df.groupby(["id", "label"])["timestamp"]
    .count()
    .reset_index(name="lifespan")
)


# Attach thresholds
lifespan["min_required"] = lifespan["label"].map(global_lifespan_thresholds)

# Identify short-lived objects
lifespan["is_outlier"] = lifespan["lifespan"] < lifespan["min_required"]

# Get outlier IDs
short_lived_ids = set(lifespan.loc[lifespan["is_outlier"], "id"].tolist())

# Drop them from the cleaned dataframe
df = df[~df["id"].isin(short_lived_ids)]

# Logging
print(f"[lifespan filter] Removed {len(short_lived_ids)} short-lived IDs")

[lifespan filter] Removed 0 short-lived IDs


In [13]:
df.reset_index(inplace = True, drop = True)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48099343 entries, 0 to 48099342
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


### 2. Footpath Zone Filtering

In [15]:
footpath_zones = [{"id":"1081","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((-8.6426068 4.1825497, -5.2591289 6.6106291, 2.4873278 -1.8810115, 3.0701297 -4.1885040, -4.8739637 -7.3121410, -5.9190415 -5.1056689, -14.9469623 -8.3614314, -15.4947729 -6.7531838, -6.4707399 -3.4197283, -5.5463181 -0.1021501, -8.6426068 4.1825497))","min_z":-1.5,"max_z":3.5},
{"id":"1082","name":"FalseDetection (Vehicle as Pedestrian)","type":"analytics","vertices":"POLYGON ((-3.3894790 -13.3168241, 11.5404031 -7.8269771, 18.9154074 -17.2223685, 14.9261910 -19.9865761, 21.4404136 -27.6760383, 20.0765345 -28.6872835, 7.1069287 -14.4841637, -11.9112838 -20.7401988, -12.4875105 -18.6473010, -2.5169133 -15.3193791, -3.3894790 -13.3168241))","min_z":-1.5,"max_z":3.5},
{"id":"1083","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((65.0702212 14.5604360, 36.5624449 3.1898636, 35.8594607 -0.3697733, 48.5353798 -17.2319024, 52.8105665 -14.6263596, 49.1754262 -9.8991876, 52.2197357 2.2111523, 68.4919414 9.0673664, 65.0702212 14.5604360))","min_z":-1.5,"max_z":3.5},
{"id":"1084","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((5.9894856 29.2443364, 7.5000886 30.3643575, 15.4508444 22.7984588, 22.8276900 37.1368332, 27.9001775 34.9101554, 20.8424398 19.4127592, 17.0919999 16.0917894, 5.9894856 29.2443364))","min_z":-1.5,"max_z":3.5}]

import geopandas as gpd
from shapely import wkt

zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

import geopandas as gpd

columns = df.columns.tolist()

def attach_zones_to_objects(df, gdf_zones, how = "inner", batch_size=100000):


    num_chunks = len(df) // batch_size + 1
    
    output_chunks = []

    for i in range(num_chunks):
        chunk = df.iloc[i*batch_size : (i+1)*batch_size].copy()

        if len(chunk) == 0:
            continue

        gdf_chunk = gpd.GeoDataFrame(
            chunk,
            geometry=gpd.points_from_xy(chunk["pos_x"], chunk["pos_y"]),
        )

        joined = gpd.sjoin(gdf_chunk, gdf_zones, how=how, predicate="within")

        # Rename columns to meaningful names
        joined = joined.rename(columns={
            "id_left": "id",
            "id_right": "zone"
        })

        joined = joined[columns + ["zone"]]

        output_chunks.append(joined)

    return pd.concat(output_chunks, ignore_index=True)


df = attach_zones_to_objects(df, gdf_zones, how = "left")

def apply_footpath_zone_filter(df):

    max_speed = {
        1: 4.0,
        2: 6.0,
        3: 12.0,
        4: 12.0,
        5: 4.0,
        6: 12.0,
        7: 0.0,
        8: 0.0,
    }

    df_zone = df[df["zone"].notnull()].copy()
    speed_limit_series = df_zone["label"].map(max_speed)

    forbidden_mask = df_zone["label"].isin(
        [7, 8]
    )

    speed_exceed_mask  = df_zone["vel"] > speed_limit_series

    remove_mask = forbidden_mask | speed_exceed_mask

    removed_ids = df_zone.loc[remove_mask, "id"].unique()
    df = df.loc[~df["id"].isin(removed_ids)].copy()

    print(f"[footpath zone] Removed {len(removed_ids)} objects")

    return df

In [16]:
df = apply_footpath_zone_filter(df)

[footpath zone] Removed 152 objects


In [17]:
df.drop(columns=["zone"], inplace=True)

In [18]:
df.reset_index(drop=True, inplace=True)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48049240 entries, 0 to 48049239
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


### 3. Crosswalk Zone Filtering

In [ ]:
crosswalk_zones = [{"id":"1015","name":"Crosswalk Houba - South","type":"analytics","vertices":"POLYGON ((25.0 -23.6, 42.8 -8.5, 40.3 -5.5, 22.6 -21.0, 25.0 -23.6))","min_z":-1.5,"max_z":3.5},
{"id":"1016","name":"Crosswalk Amandiers","type":"analytics","vertices":"POLYGON ((-2.7 -13.4, -4.8 -7.4, -1.4 -6.0, 0.7 -12.2, -2.7 -13.4))","min_z":-1.5,"max_z":3.5},
{"id":"1017","name":"Crosswalk Houba - North","type":"analytics","vertices":"POLYGON ((-3.1 4.7, 13.8 19.3, 16.9 16.1, 0.0 1.4, -3.1 4.7))","min_z":-1.5,"max_z":3.5},
{"id":"1018","name":"Crosswalk Magnolias","type":"analytics","vertices":"POLYGON ((30.5 15.5, 32.2 18.9, 23.4 23.5, 21.6 20.2, 30.5 15.5))","min_z":-1.5,"max_z":3.5},
{"id":"1019","name":"Crosswalk Charlotte [1]","type":"analytics","vertices":"POLYGON ((36.4 15.3, 40.4 5.2, 37.1 3.8, 33.1 14.0, 36.4 15.3))","min_z":-1.5,"max_z":3.5}]

zones_df = pd.DataFrame(crosswalk_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

# ---------------------------------------------
# 4. Compute polygon orientation (longest edge)
# ---------------------------------------------
def compute_polygon_orientation(poly):
    coords = list(poly.exterior.coords)
    edges = []

    for (x1, y1), (x2, y2) in zip(coords, coords[1:]):
        dx, dy = x2 - x1, y2 - y1
        length = (dx**2 + dy**2) ** 0.5
        edges.append((length, dx, dy))

    # longest edge
    _, dx, dy = max(edges, key=lambda e: e[0])
    return np.degrees(np.arctan2(dy, dx))
Y

# ---------------------------------------------
# 5. Apply zone-level filtering for cars
# ---------------------------------------------
def filter_parallel_cars(df_zone, orientation_deg, threshold=4.0):
    """
    df_zone contains objects inside a single crosswalk zone
    """

    cars = df_zone[df_zone["label"] > 2].copy()  # label=4 = car

    if cars.empty:
        return [], df_zone  # nothing removed

    # Compute heading
    cars["heading_deg"] = np.degrees(cars["yaw"])

    # Fallback if yaw is missing
    missing = cars["heading_deg"].isna()
    cars.loc[missing, "heading_deg"] = np.degrees(
        np.arctan2(cars.loc[missing, "vel_y"], cars.loc[missing, "vel_x"])
    )

    # Angular difference
    def angle_diff(a, b):
        d = (a - b + 180) % 360 - 180
        return abs(d)

    cars["angle_diff"] = cars.apply(
        lambda r: angle_diff(r["heading_deg"], orientation_deg),
        axis=1
    )

    # Cars moving parallel to crosswalk axis
    parallel = cars[cars["angle_diff"] < threshold]["id"].unique().tolist()

    # Remove them
    df_zone_filtered = df_zone[~df_zone["id"].isin(parallel)]

    return parallel, df_zone_filtered


# Precompute zone orientations
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

removed_ids_global = []

df = attach_zones_to_objects(df, gdf_zones, how = "left")

# Process each zone separately
for zone_id, zone_df in df.groupby("zone"): 
    print(f"Processing crosswalk zone {zone_id} with {len(zone_df)} objects")

    zone_orientation = float(
        gdf_zones.loc[gdf_zones["id"] == str(zone_id), "orientation_deg"].iloc[0]
    )

    removed_ids, zone_filtered = filter_parallel_cars(
        zone_df, orientation_deg=zone_orientation, threshold=4.0
    )

    removed_ids_global.append(removed_ids)

    print(f"[crosswalk zone] Zone {zone_id}: Removed {len(removed_ids)} parallel cars")

removed_ids_global = [item for sublist in removed_ids_global for item in sublist]
df = df[~df["id"].isin(removed_ids_global)]

MemoryError: Unable to allocate 366. MiB for an array with shape (48021479, 1) and data type object

In [21]:
removed_ids_global

[11133360,
 11181164,
 11546706,
 11689801,
 11708598,
 11826070,
 11865718,
 11992093,
 11602740,
 11617405,
 11711335,
 11302588,
 11307015,
 11541183,
 11619109,
 11749041,
 11799635]

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48021479 entries, 0 to 48021478
Data columns (total 16 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
 15  zone       object        
dtypes: datetime64[ns](1), float32(12), int32(2), object(1)
memory usage: 3.2+ GB


In [ ]:
df.drop(columns = ['zone'], inplace = True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48021479 entries, 0 to 48021478
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


### 4. Remove All Lifetime Static Objects

In [ ]:
STATIC_THRESHOLD = 0.5     # m/s
STATIC_RATIO_MIN = 0.8     # 80% of frames static


# --- Build per-object velocity history ---
df_vel = (
    df.groupby(["id", "label"])["vel"]
    .apply(list)
    .reset_index()
)

# Compute lifespan
df_vel["lifespan"] = df_vel["vel"].apply(len)

# Static frame count
df_vel["static_frames"] = df_vel["vel"].apply(
    lambda v: sum(vi < STATIC_THRESHOLD for vi in v)
)

# Static ratio
df_vel["static_ratio"] = df_vel["static_frames"] / df_vel["lifespan"]

# Static vehicle flag
df_vel["is_static_vehicle"] = df_vel["static_ratio"] >= STATIC_RATIO_MIN

# -----------------------------
# Select only static heavy cars
# -----------------------------
removable_static_ids = set(
    df_vel[
        (df_vel["is_static_vehicle"])
        # (df_vel["label"].isin(STATIC_LABELS))
    ]["id"].astype(int).tolist()
)

print(f"Static removable vehicle IDs: {len(removable_static_ids)}")

df = df[~df['id'].isin(removable_static_ids)]

Static removable vehicle IDs: 10295


In [ ]:
df.reset_index(inplace = True, drop = True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22749123 entries, 0 to 22749122
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 1.4 GB


### 5. Attach Near miss zones

In [ ]:
near_miss_zones = [{"id":"1074","name":"Pedestrian Crossing 1","type":"analytics","vertices":"POLYGON ((24.2738305 11.0336169, 31.7356793 13.5560292, 36.2078141 15.1576709, 43.7450150 17.4969908, 47.2622737 8.1635769, 36.9982138 3.5232094, 32.5602804 1.6666404, 27.3267870 -0.3062503, 24.2738305 11.0336169))","min_z":-1.5,"max_z":3.5},
{"id":"1075","name":"Pedestrian Crossing 2","type":"analytics","vertices":"POLYGON ((28.4322135 35.4097216, 37.6101806 31.0730297, 29.7613022 13.8943178, 24.0457174 12.0937765, 19.1230952 14.6240664, 28.4322135 35.4097216))","min_z":-1.5,"max_z":3.5},
{"id":"1076","name":"Pedestrian Crossing 3","type":"analytics","vertices":"POLYGON ((-0.4008521 23.7028977, 3.8500930 26.7534852, 11.9475235 17.4708113, 13.9290183 18.9399734, 22.7019680 11.3504234, 15.4719481 5.8276760, -0.4008521 23.7028977))","min_z":-1.5,"max_z":3.5},
{"id":"1077","name":"Pedestrian Crossing 4","type":"analytics","vertices":"POLYGON ((-13.4857933 16.0361501, -2.3947097 3.8495495, 2.2686744 -1.0889511, 11.2978547 7.4421036, 7.8841631 11.7104214, 5.8591757 11.8061846, -5.7526480 23.8885602, -13.4857933 16.0361501))","min_z":-1.5,"max_z":3.5},
{"id":"1078","name":"Pedestrian Crossing 5","type":"analytics","vertices":"POLYGON ((3.3656059 -4.4085345, -24.1838846 -14.1766537, -22.1521598 -19.7115374, 6.1376057 -10.0653907, 3.3656059 -4.4085345))","min_z":-1.5,"max_z":3.5},
{"id":"1079","name":"Pedestrian Crossing 6","type":"analytics","vertices":"POLYGON ((18.1027860 -15.9690219, 24.7117391 -12.1747441, 30.5032592 -18.9617192, 28.5317011 -21.0059641, 40.8463196 -36.3820050, 37.9033636 -39.0948896, 18.1027860 -15.9690219))","min_z":-1.5,"max_z":3.5},
{"id":"1080","name":"Pedestrian Crossing 7","type":"analytics","vertices":"POLYGON ((25.6180338 -11.6782049, 42.4498255 -32.1848405, 52.3394447 -23.7967882, 36.5273842 -0.7475620, 25.6180338 -11.6782049))","min_z":-1.5,"max_z":3.5},
]

zones_df = pd.DataFrame(near_miss_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

def attach_zones_to_objects(df, gdf_zones, how = "inner", batch_size=100000):


    num_chunks = len(df) // batch_size + 1
    
    output_chunks = []

    for i in range(num_chunks):
        chunk = df.iloc[i*batch_size : (i+1)*batch_size].copy()

        if len(chunk) == 0:
            continue

        gdf_chunk = gpd.GeoDataFrame(
            chunk,
            geometry=gpd.points_from_xy(chunk["pos_x"], chunk["pos_y"]),
        )

        joined = gpd.sjoin(gdf_chunk, gdf_zones, how=how, predicate="within")

        # Rename columns to meaningful names
        joined = joined.rename(columns={
            "id_left": "id",
            "id_right": "zone"
        })

        joined = joined[columns + ["zone"]]

        output_chunks.append(joined)

    return pd.concat(output_chunks, ignore_index=True)

df = attach_zones_to_objects(df, gdf_zones)

df.reset_index(inplace = True, drop = True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6504009 entries, 0 to 6504008
Data columns (total 16 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
 15  zone       object        
dtypes: datetime64[ns](1), float32(12), int32(2), object(1)
memory usage: 446.6+ MB


In [ ]:
df.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel,zone
0,2025-06-02 00:00:09.430505037,11077156,6,-8.820312,19.593750,0.070007,5.835938,2.320312,2.619141,9.242188,-8.421875,0.0,5.531250,0.099976,12.960712,1077
1,2025-06-02 00:00:09.534637928,11077156,6,-7.859375,18.765625,0.070007,5.835938,2.320312,2.619141,9.382812,-8.359375,0.0,5.535156,0.109985,12.885175,1077
2,2025-06-02 00:00:09.623222113,11077156,6,-6.929688,17.906250,0.040009,5.835938,2.320312,2.619141,9.296875,-8.257812,0.0,5.535156,0.099976,12.972755,1077
3,2025-06-02 00:00:09.735569954,11077156,6,-6.019531,17.093750,0.040009,5.835938,2.320312,2.619141,9.382812,-8.218750,0.0,5.542969,0.109985,12.770443,1077
4,2025-06-02 00:00:09.839029074,11077156,6,-5.109375,16.250000,-0.029999,5.835938,2.320312,2.619141,9.359375,-8.289062,0.0,5.546875,0.080017,12.696608,1077


### 6. DRAC Near-Miss Analysis

In [ ]:
import time

In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple, Optional

class VectorizedPedestrianVehicleDRACDetector:
    """
    Highly optimized DRAC Detection System for Pedestrian-Vehicle conflicts
    Uses vectorized operations to process millions of records efficiently
    """
    
    # Label definitions
    PEDESTRIAN_LABEL = 1
    VEHICLE_LABELS = [3, 4, 6, 7, 8]  # motorcycle, car, van, truck, bus
    
    LABEL_NAMES = {
        1: 'pedestrian', 2: 'bicycle', 3: 'motorcycle', 
        4: 'car', 5: 'escooter', 6: 'van', 7: 'truck', 8: 'bus'
    }
    
    # Safe stopping distances for pedestrians (meters)
    SAFE_STOPPING_DISTANCE = {
        3: 3.0,   # motorcycle
        4: 4.0,   # car
        6: 5.0,   # van
        7: 7.0,   # truck
        8: 8.0    # bus
    }
    
    # DRAC thresholds for pedestrian conflicts
    DRAC_THRESHOLD_MODERATE = 2.0
    DRAC_THRESHOLD_SERIOUS = 4.0
    DRAC_THRESHOLD_CRITICAL = 7.0
    
    # Pedestrian conflict-specific parameters
    PEDESTRIAN_REACTION_ZONE = 2.0
    
    def __init__(self, 
                 drac_threshold: float = 2.0,
                 approach_angle_threshold: float = np.pi/3,  # 60 degrees
                 max_distance: float = 10.0,
                 min_vehicle_speed: float = 2.0,
                 chunk_size: int = 100000):
        """
        Initialize Vectorized Pedestrian-Vehicle DRAC detector
        
        Args:
            drac_threshold: Minimum DRAC value to report (m/s²)
            approach_angle_threshold: Maximum angle for vehicle approaching pedestrian
            max_distance: Maximum distance to check for conflicts (meters)
            min_vehicle_speed: Minimum vehicle speed to consider (m/s)
            chunk_size: Number of timestamps to process at once
        """
        self.drac_threshold = drac_threshold
        self.approach_angle_threshold = approach_angle_threshold
        self.max_distance = max_distance
        self.min_vehicle_speed = min_vehicle_speed
        self.chunk_size = chunk_size
        self.cos_angle_threshold = np.cos(approach_angle_threshold)
        
    def process_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Process entire dataframe using vectorized operations
        
        Args:
            df: DataFrame with object detection data
            
        Returns:
            DataFrame with pedestrian-vehicle near-miss events
        """
        print(f"Processing {len(df)} records...")
        print(f"Unique timestamps: {df['timestamp'].nunique()}")
        
        # Get unique timestamps and process in chunks
        unique_timestamps = df['timestamp'].unique()
        total_timestamps = len(unique_timestamps)
        
        all_results = []
        analytic_id = 0
        
        for chunk_start in range(0, total_timestamps, self.chunk_size):
            chunk_end = min(chunk_start + self.chunk_size, total_timestamps)
            timestamp_chunk = unique_timestamps[chunk_start:chunk_end]
            
            print(f"Processing timestamps {chunk_start+1}-{chunk_end} of {total_timestamps}...", end='\r')
            
            # Get data for this chunk of timestamps
            chunk_df = df[df['timestamp'].isin(timestamp_chunk)]
            
            # Process this chunk
            chunk_results = self._process_chunk_vectorized(chunk_df, analytic_id)
            
            if len(chunk_results) > 0:
                print(f"  ✓ Found {len(chunk_results)} near-miss events in this chunk")
                all_results.append(chunk_results)
                analytic_id = chunk_results['analytic_id'].max() + 1
        
        print()  # New line after progress
        
        # Combine all results
        if all_results:
            result_df = pd.concat(all_results, ignore_index=True)
            print(f"✓ Detected {len(result_df)} near-miss events")
            return result_df
        else:
            print("✓ No near-miss events detected")
            return self._empty_result_dataframe()
    
    def _process_chunk_vectorized(self, chunk_df: pd.DataFrame, start_analytic_id: int) -> pd.DataFrame:
        """
        Process a chunk of data using vectorized operations
        """
        # Separate pedestrians and vehicles
        pedestrians = chunk_df[chunk_df['label'] == self.PEDESTRIAN_LABEL].copy()
        vehicles = chunk_df[chunk_df['label'].isin(self.VEHICLE_LABELS)].copy()
        
        if len(pedestrians) == 0 or len(vehicles) == 0:
            return self._empty_result_dataframe()
        
        # Filter vehicles by speed
        vehicles = vehicles[vehicles['vel'] >= self.min_vehicle_speed]
        
        if len(vehicles) == 0:
            return self._empty_result_dataframe()
        
        # Create cartesian product by timestamp
        # Merge on timestamp to pair pedestrians and vehicles
        pedestrians['_merge_key'] = 1
        vehicles['_merge_key'] = 1
        
        pairs = pd.merge(
            pedestrians,
            vehicles,
            on=['timestamp', '_merge_key'],
            suffixes=('_ped', '_veh')
        ).drop('_merge_key', axis=1)
        
        if len(pairs) == 0:
            return self._empty_result_dataframe()
        
        # Vectorized calculations
        # Calculate relative positions
        pairs['delta_x'] = pairs['pos_x_veh'] - pairs['pos_x_ped']
        pairs['delta_y'] = pairs['pos_y_veh'] - pairs['pos_y_ped']
        pairs['distance'] = np.sqrt(pairs['delta_x']**2 + pairs['delta_y']**2)
        
        # Filter by distance
        pairs = pairs[(pairs['distance'] > 0.5) & (pairs['distance'] <= self.max_distance)]
        
        if len(pairs) == 0:
            return self._empty_result_dataframe()
        
        # Normalize direction vectors (vehicle to pedestrian)
        pairs['to_ped_x'] = -pairs['delta_x'] / pairs['distance']
        pairs['to_ped_y'] = -pairs['delta_y'] / pairs['distance']
        
        # Vehicle heading from yaw
        pairs['veh_heading_x'] = np.cos(pairs['yaw_veh'])
        pairs['veh_heading_y'] = np.sin(pairs['yaw_veh'])
        
        # Calculate alignment (dot product)
        pairs['alignment'] = (pairs['veh_heading_x'] * pairs['to_ped_x'] + 
                             pairs['veh_heading_y'] * pairs['to_ped_y'])
        
        # Filter by approach angle
        pairs = pairs[pairs['alignment'] > self.cos_angle_threshold]
        
        if len(pairs) == 0:
            return self._empty_result_dataframe()
        
        # Calculate closing speed
        pairs['closing_speed'] = (pairs['vel_x_veh'] * pairs['to_ped_x'] + 
                                 pairs['vel_y_veh'] * pairs['to_ped_y'])
        
        # Filter: must be closing in
        pairs = pairs[pairs['closing_speed'] > 0]
        
        if len(pairs) == 0:
            return self._empty_result_dataframe()
        
        # Calculate effective distance with safety buffers
        pairs['safe_distance'] = pairs['label_veh'].map(self.SAFE_STOPPING_DISTANCE).fillna(4.0)
        pairs['vehicle_buffer'] = pairs['size_x_veh'] / 2.0
        pairs['ped_buffer'] = pairs[['size_x_ped', 'size_y_ped']].max(axis=1) / 2.0
        pairs['total_buffer'] = (pairs['vehicle_buffer'] + 
                                pairs['ped_buffer'] + 
                                pairs['safe_distance'] + 
                                self.PEDESTRIAN_REACTION_ZONE)
        pairs['effective_distance'] = pairs['distance'] - pairs['total_buffer']
        
        # Calculate DRAC
        # For already too close cases
        pairs['drac'] = np.where(
            pairs['effective_distance'] <= 0,
            999.9,  # Cap at max value
            np.minimum((pairs['closing_speed'] ** 2) / (2.0 * pairs['effective_distance']), 999.9)
        )
        
        # Calculate TTC
        pairs['ttc'] = np.minimum(pairs['distance'] / pairs['closing_speed'], 999.9)
        
        # Determine severity
        pairs['severity'] = 'low'
        pairs.loc[(pairs['drac'] >= self.DRAC_THRESHOLD_MODERATE) | (pairs['ttc'] < 4.0), 'severity'] = 'moderate'
        pairs.loc[(pairs['drac'] >= self.DRAC_THRESHOLD_SERIOUS) | (pairs['ttc'] < 2.5), 'severity'] = 'serious'
        pairs.loc[(pairs['drac'] >= self.DRAC_THRESHOLD_CRITICAL) | (pairs['ttc'] < 1.5), 'severity'] = 'critical'
        
        # Filter by threshold
        pairs = pairs[(pairs['drac'] >= self.drac_threshold) | (pairs['ttc'] <= 5.0)]
        
        if len(pairs) == 0:
            return self._empty_result_dataframe()
        
        # Calculate relative velocity
        pairs['rel_vel_x'] = pairs['vel_x_veh'] - pairs['vel_x_ped']
        pairs['rel_vel_y'] = pairs['vel_y_veh'] - pairs['vel_y_ped']
        pairs['rel_vel'] = np.sqrt(pairs['rel_vel_x']**2 + pairs['rel_vel_y']**2)
        
        # Create object pair labels
        pairs['vehicle_name'] = pairs['label_veh'].map(self.LABEL_NAMES)
        pairs['object_pair_labels'] = 'pedestrian-' + pairs['vehicle_name']
        
        # Add analytic IDs
        pairs['analytic_id'] = range(start_analytic_id, start_analytic_id + len(pairs))
        
        # Select and rename columns to match output schema
        result = pairs[[
            'timestamp',
            'analytic_id',
            'id_ped',
            'id_veh',
            'label_ped',
            'label_veh',
            'object_pair_labels',
            'pos_x_ped',
            'pos_y_ped',
            'pos_x_veh',
            'pos_y_veh',
            'vel_x_ped',
            'vel_y_ped',
            'vel_ped',
            'vel_x_veh',
            'vel_y_veh',
            'vel_veh',
            'yaw_ped',
            'yaw_veh',
            'size_x_ped',
            'size_y_ped',
            'size_x_veh',
            'size_y_veh',
            'distance',
            'rel_vel',
            'drac',
            'severity',
            'ttc',
            'closing_speed',
            'effective_distance'
        ]].copy()
        
        # Rename columns to match output schema
        result.columns = [
            'timestamp',
            'analytic_id',
            'id_obj1',
            'id_obj2',
            'label_obj1',
            'label_obj2',
            'object_pair_labels',
            'pos_x_obj1',
            'pos_y_obj1',
            'pos_x_obj2',
            'pos_y_obj2',
            'vel_x_obj1',
            'vel_y_obj1',
            'vel_obj1',
            'vel_x_obj2',
            'vel_y_obj2',
            'vel_obj2',
            'yaw_obj1',
            'yaw_obj2',
            'size_x_obj1',
            'size_y_obj1',
            'size_x_obj2',
            'size_y_obj2',
            'rel_dist',
            'rel_vel',
            'drac',
            'severity',
            'ttc',
            'closing_speed',
            'effective_distance'
        ]
        
        return result
    
    def _empty_result_dataframe(self) -> pd.DataFrame:
        """Return empty DataFrame with correct schema"""
        return pd.DataFrame(columns=[
            'timestamp', 'analytic_id', 'id_obj1', 'id_obj2', 'label_obj1',
            'label_obj2', 'object_pair_labels', 'pos_x_obj1', 'pos_y_obj1',
            'pos_x_obj2', 'pos_y_obj2', 'vel_x_obj1', 'vel_y_obj1', 'vel_obj1',
            'vel_x_obj2', 'vel_y_obj2', 'vel_obj2', 'yaw_obj1', 'yaw_obj2',
            'size_x_obj1', 'size_y_obj1', 'size_x_obj2', 'size_y_obj2', 
            'rel_dist', 'rel_vel', 'drac', 'severity', 'ttc', 'closing_speed',
            'effective_distance'
        ])

import time

print("="*80)
print("VECTORIZED PEDESTRIAN-VEHICLE DRAC NEAR-MISS DETECTION")
print("="*80)

# Create detector
detector = VectorizedPedestrianVehicleDRACDetector(
    drac_threshold=2.0,
    approach_angle_threshold=np.pi/3,
    max_distance=10.0,
    min_vehicle_speed=2.0,
    chunk_size=50  # Process 50 timestamps at a time
)


df_1 = df.iloc[200000:300000]

print(f"Dataset: {len(df_1)} rows")
print(f"Pedestrians: {len(df_1[df_1['label']==1])}")
print(f"Vehicles: {len(df_1[df_1['label'].isin([3,4,6,7,8])])}")

# Process
start_time = time.time()
results = detector.process_dataframe(df_1)
elapsed = time.time() - start_time

print(f"\nProcessing time: {elapsed:.2f} seconds")
print(f"Speed: {len(df)/elapsed:.0f} objects/second")

if len(results) > 0:
    print(f"\nDetected {len(results)} near-miss events:")
    print(results['severity'].value_counts().to_dict())
    print(f"\nSample results:")
    print(results[['timestamp', 'object_pair_labels', 'rel_dist', 'drac', 'ttc', 'severity']].head())

print("\n" + "="*80)
print("PERFORMANCE TIPS FOR YOUR 48M DATASET:")
print("="*80)
print("1. Process in chunks - adjust chunk_size based on your RAM")
print("2. Filter data before processing (e.g., by location, time of day)")
print("3. Use smaller max_distance if appropriate (reduces pairs to check)")
print("4. Consider parallel processing with Dask or multiprocessing")
print("5. Save intermediate results to disk during processing")
print("="*80)

VECTORIZED PEDESTRIAN-VEHICLE DRAC NEAR-MISS DETECTION
Dataset: 100000 rows
Pedestrians: 9063
Vehicles: 88804
Processing 100000 records...
Unique timestamps: 15130
  ✓ Found 5 near-miss events in this chunk
  ✓ Found 3 near-miss events in this chunk.
  ✓ Found 6 near-miss events in this chunk.
  ✓ Found 3 near-miss events in this chunk.
  ✓ Found 16 near-miss events in this chunk
  ✓ Found 3 near-miss events in this chunk.
  ✓ Found 3 near-miss events in this chunk.
  ✓ Found 21 near-miss events in this chunk
  ✓ Found 16 near-miss events in this chunk
  ✓ Found 6 near-miss events in this chunk.


  ✓ Found 5 near-miss events in this chunk.
  ✓ Found 2 near-miss events in this chunk.
  ✓ Found 4 near-miss events in this chunk.
  ✓ Found 9 near-miss events in this chunk.
  ✓ Found 7 near-miss events in this chunk.
  ✓ Found 2 near-miss events in this chunk.
  ✓ Found 3 near-miss events in this chunk.
  ✓ Found 4 near-miss events in this chunk.
  ✓ Found 17 near-miss events in this chunk
  ✓ Found 2 near-miss events in this chunk.
  ✓ Found 1 near-miss events in this chunk.
  ✓ Found 6 near-miss events in this chunk...
  ✓ Found 2 near-miss events in this chunk...
  ✓ Found 5 near-miss events in this chunk...
  ✓ Found 1 near-miss events in this chunk...
  ✓ Found 10 near-miss events in this chunk..
  ✓ Found 5 near-miss events in this chunk...
  ✓ Found 10 near-miss events in this chunk..
  ✓ Found 8 near-miss events in this chunk...
  ✓ Found 9 near-miss events in this chunk...
Processing timestamps 15101-15130 of 15130...
✓ Detected 194 near-miss events

Processing time: 2.19 s

In [ ]:
results[results['severity'] == 'critical']

,timestamp,analytic_id,id_obj1,id_obj2,label_obj1,label_obj2,object_pair_labels,pos_x_obj1,pos_y_obj1,pos_x_obj2,...,size_y_obj1,size_x_obj2,size_y_obj2,rel_dist,rel_vel,drac,severity,ttc,closing_speed,effective_distance
0,2025-06-03 04:00:27.186928988,0,12081860,12081961,1,4,pedestrian-car,24.531250,21.046875,28.421875,...,0.509766,4.371094,1.919922,9.840814,4.561757,10.687935,critical,1.798645,5.471236,1.400384
1,2025-06-03 04:00:27.285342932,1,12081860,12081961,1,4,pedestrian-car,24.687500,21.093750,28.406250,...,0.509766,4.371094,1.919922,9.615410,4.601838,12.938895,critical,1.743769,5.514153,1.174980
2,2025-06-03 04:00:27.462781906,2,12081860,12081961,1,4,pedestrian-car,24.437500,21.296875,28.328125,...,0.509766,4.371094,1.919922,8.518506,4.746484,188.614225,critical,1.569645,5.427027,0.078076
3,2025-06-03 04:00:27.563081980,3,12081860,12081961,1,4,pedestrian-car,24.203125,21.328125,28.375000,...,0.509766,4.371094,1.919922,8.148898,4.830078,999.900000,critical,1.531052,5.322418,-0.291532
4,2025-06-03 04:00:27.676052094,4,12081860,12081961,1,4,pedestrian-car,24.062500,21.375000,28.453125,...,0.509766,4.371094,1.919922,7.805153,4.882214,999.900000,critical,1.510916,5.165842,-0.635277
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,2025-06-03 04:23:28.044661045,189,12094318,12094122,1,6,pedestrian-van,42.875000,6.480469,34.750000,...,0.609863,5.210938,2.130859,8.156729,4.406932,999.900000,critical,1.636067,4.985571,-1.753672
190,2025-06-03 04:23:28.145623922,190,12094318,12094122,1,6,pedestrian-van,43.062500,6.488281,35.281250,...,0.609863,5.210938,2.130859,7.796555,4.483590,999.900000,critical,1.531289,5.091500,-2.113845
191,2025-06-03 04:23:28.247948885,191,12094318,12094122,1,6,pedestrian-van,43.250000,6.500000,35.875000,...,0.609863,5.210938,2.130859,7.378849,4.717316,999.900000,critical,1.373253,5.373260,-2.531552
192,2025-06-03 04:23:28.344271898,192,12094318,12094122,1,6,pedestrian-van,43.406250,6.511719,36.437500,...,0.609863,5.210938,2.130859,6.968760,4.989475,999.900000,critical,1.232856,5.652532,-2.941640


### 7. Observed Decceleration Method

In [ ]:
from typing import Tuple, Optional, Dict
import numpy as np
import pandas as pd
import time


class PedestrianVehicleDecelerationDetector:
    """
    Near-miss detection based on observed vehicle deceleration

    Detects scenarios where:
    1. Vehicle is close to pedestrian (< max_distance)
    2. Vehicle is moving above speed threshold
    3. Vehicle is heading toward pedestrian
    4. Vehicle decelerates in the next few seconds

    Severity is classified based on deceleration magnitude
    """

    # Label definitions
    PEDESTRIAN_LABEL = 1
    VEHICLE_LABELS = [4, 6, 7, 8]  # car, van, truck, bus

    LABEL_NAMES = {
        1: 'pedestrian', 2: 'bicycle', 3: 'motorcycle',
        4: 'car', 5: 'escooter', 6: 'van', 7: 'truck', 8: 'bus'
    }

    # Deceleration thresholds (m/s²)
    DECEL_THRESHOLD_MODERATE = 2.0   # Noticeable braking
    DECEL_THRESHOLD_SERIOUS = 4.0    # Hard braking
    DECEL_THRESHOLD_CRITICAL = 6.5   # Emergency braking

    def __init__(self,
                 max_pair_distance: float = 1.0,
                 min_vehicle_speed: float = 3.0,
                 approach_angle_threshold: float = np.pi/3,  # 60 degrees
                 observation_window: float = 3.0,  # seconds
                 min_deceleration: float = 1.5,  # kept but not used for filtering now
                 sampling_rate: float = 0.1,  # 10Hz = 0.1s per frame
                 chunk_size: int = 10000):
        """
        Initialize Deceleration-Based Near-Miss Detector

        Args:
            max_pair_distance: Maximum distance to consider (meters)
            min_vehicle_speed: Minimum vehicle speed to check (m/s)
            approach_angle_threshold: Maximum angle for approaching (radians)
            observation_window: Time to observe vehicle behavior (seconds)
            min_deceleration: Minimum deceleration to report (m/s²)
            sampling_rate: Time between frames (seconds), 0.1s for 10Hz
            chunk_size: Number of timestamps to process at once
        """
        self.max_pair_distance = max_pair_distance
        self.min_vehicle_speed = min_vehicle_speed
        self.approach_angle_threshold = approach_angle_threshold
        self.observation_window = observation_window
        self.min_deceleration = min_deceleration
        self.sampling_rate = sampling_rate
        self.chunk_size = chunk_size
        self.cos_angle_threshold = np.cos(approach_angle_threshold)

        # Calculate how many frames to look ahead
        self.frames_to_observe = int(observation_window / sampling_rate)

    def process_dataframe(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Process dataframe to detect deceleration-based near-misses

        Args:
            df: DataFrame with object detection data

        Returns:
            all_candidates_df: all pedestrian-vehicle pairs after spatial/angle filters
            result_df: DataFrame with deceleration metrics and severity for each pair
        """
        print(f"Processing {len(df)} records...")
        print(f"Observation window: {self.observation_window}s ({self.frames_to_observe} frames)")

        # Sort by timestamp to ensure temporal order
        df = df.sort_values('timestamp').reset_index(drop=True)

        # Get unique timestamps
        unique_timestamps = df['timestamp'].unique()
        total_timestamps = len(unique_timestamps)

        # We can only analyze timestamps that have enough future data
        max_analyzable = total_timestamps - self.frames_to_observe

        if max_analyzable <= 0:
            print("Not enough data for observation window")
            return pd.DataFrame(), self._empty_result_dataframe()

        print(f"Analyzable timestamps: {max_analyzable} of {total_timestamps}")

        all_results = []
        all_candidates = []

        # Process in chunks
        for chunk_start in range(0, max_analyzable, self.chunk_size):
            chunk_end = min(chunk_start + self.chunk_size, max_analyzable)
            timestamp_chunk = unique_timestamps[chunk_start:chunk_end]

            print(f"Processing timestamps {len(timestamp_chunk)} of {max_analyzable}...", end='\r')

            # Get initial data for this chunk
            chunk_df = df[df['timestamp'].isin(timestamp_chunk)]

            # Get future data for observation window
            future_timestamps = unique_timestamps[chunk_start:chunk_end + self.frames_to_observe]
            extended_df = df[df['timestamp'].isin(future_timestamps)]

            # Process this chunk
            cand_chunk, chunk_results = self._process_chunk_with_observation(
                chunk_df, extended_df, unique_timestamps[chunk_start:]
            )

            if len(cand_chunk) > 0:
                all_candidates.append(cand_chunk)

            if len(chunk_results) > 0:
                all_results.append(chunk_results)

        print()

        # Combine candidates
        if all_candidates:
            all_candidates_df = pd.concat(all_candidates, ignore_index=True)
        else:
            all_candidates_df = pd.DataFrame()

        # Combine results
        if all_results:
            result_df = pd.concat(all_results, ignore_index=True)
            print(f"✓ Produced {len(result_df)} events with deceleration metrics")
            return all_candidates_df, result_df
        else:
            print("✓ No events with valid trajectories")
            return all_candidates_df, self._empty_result_dataframe()

    def _process_chunk_with_observation(self,
                                        initial_df: pd.DataFrame,
                                        extended_df: pd.DataFrame,
                                        timestamps: np.ndarray) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Process chunk and observe vehicle behavior over time
        """
        # Step 1: Find initial pedestrian-vehicle pairs meeting criteria
        candidate_pairs = self._find_candidate_pairs(initial_df)

        if len(candidate_pairs) == 0:
            return pd.DataFrame(), self._empty_result_dataframe()

        # Step 2: Track vehicles over observation window
        results = self._observe_vehicle_deceleration(
            candidate_pairs, extended_df, timestamps
        )

        return candidate_pairs, results

    def _find_candidate_pairs(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Find pedestrian-vehicle pairs meeting initial criteria
        """
        # Separate pedestrians and vehicles
        pedestrians = df[df['label'] == self.PEDESTRIAN_LABEL].copy()
        vehicles = df[df['label'].isin(self.VEHICLE_LABELS)].copy()

        if len(pedestrians) == 0 or len(vehicles) == 0:
            return pd.DataFrame()

        # Filter vehicles by speed
        vehicles = vehicles[vehicles['vel'] >= self.min_vehicle_speed]

        if len(vehicles) == 0:
            return pd.DataFrame()

        # Create pairs by timestamp
        pedestrians['_merge_key'] = 1
        vehicles['_merge_key'] = 1

        pairs = pd.merge(
            pedestrians,
            vehicles,
            on=['timestamp', '_merge_key'],
            suffixes=('_ped', '_veh')
        ).drop('_merge_key', axis=1)

        if len(pairs) == 0:
            return pd.DataFrame()

        # Calculate distances
        pairs['delta_x'] = pairs['pos_x_veh'] - pairs['pos_x_ped']
        pairs['delta_y'] = pairs['pos_y_veh'] - pairs['pos_y_ped']
        pairs['distance'] = np.sqrt(pairs['delta_x']**2 + pairs['delta_y']**2)

        # Filter by max distance
        pairs = pairs[(pairs['distance'] > 0.5) & (pairs['distance'] <= self.max_pair_distance)]

        if len(pairs) == 0:
            return pd.DataFrame()

        # Calculate approach angle
        pairs['to_ped_x'] = -pairs['delta_x'] / pairs['distance']
        pairs['to_ped_y'] = -pairs['delta_y'] / pairs['distance']
        pairs['veh_heading_x'] = np.cos(pairs['yaw_veh'])
        pairs['veh_heading_y'] = np.sin(pairs['yaw_veh'])
        pairs['alignment'] = (pairs['veh_heading_x'] * pairs['to_ped_x'] +
                              pairs['veh_heading_y'] * pairs['to_ped_y'])

        # Filter by approach angle
        pairs = pairs[pairs['alignment'] > self.cos_angle_threshold]

        if len(pairs) == 0:
            return pd.DataFrame()

        # Store initial velocity for later deceleration calculation
        pairs['initial_vel'] = pairs['vel_veh']
        pairs['initial_vel_x'] = pairs['vel_x_veh']
        pairs['initial_vel_y'] = pairs['vel_y_veh']

        if 'zone_ped' in pairs.columns:
            pairs['zone'] = pairs['zone_ped']
        elif 'zone' in pairs.columns:
            # Keep existing zone column if present
            pass
        else:
            pairs['zone'] = 'unknown'  # Fallback

        return pairs

    def _observe_vehicle_deceleration(self,
                                      candidate_pairs: pd.DataFrame,
                                      extended_df: pd.DataFrame,
                                      timestamps: np.ndarray) -> pd.DataFrame:
        """
        Observe vehicles over time and calculate deceleration
        """
        results = []

        print(f"Observing {len(candidate_pairs)} candidate pairs for deceleration...", end='\r')

        # Group candidates by timestamp and vehicle ID
        for (timestamp, veh_id), group in candidate_pairs.groupby(['timestamp', 'id_veh']):
            # Get vehicle data for the observation window
            timestamp_idx = np.where(timestamps == timestamp)[0]
            if len(timestamp_idx) == 0:
                continue

            timestamp_idx = timestamp_idx[0]
            future_timestamps = timestamps[timestamp_idx:timestamp_idx + self.frames_to_observe + 1]

            # Get vehicle trajectory
            vehicle_trajectory = extended_df[
                (extended_df['id'] == veh_id) &
                (extended_df['timestamp'].isin(future_timestamps))
            ].sort_values('timestamp')

            # Require at least 2 samples to define a window
            if len(vehicle_trajectory) < 2:
                print(f" Skipping pair at {timestamp} vehicle {veh_id}: insufficient trajectory data", end='\r')
                continue

            # Calculate deceleration over the observation window
            decel_results = self._calculate_deceleration_metrics(
                vehicle_trajectory, group
            )

            if decel_results is not None:
                results.append(decel_results)

        if len(results) == 0:
            return self._empty_result_dataframe()

        # Convert to DataFrame
        result_df = pd.DataFrame(results)

        # No filtering by min_deceleration here

        # Add analytic IDs
        result_df['analytic_id'] = result_df['zone']

        # Classify severity (including non-deceleration)
        result_df['severity'] = result_df['max_deceleration'].apply(self._classify_severity)

        return result_df

    def _calculate_deceleration_metrics(self,
                                        trajectory: pd.DataFrame,
                                        initial_pair: pd.DataFrame) -> Optional[Dict]:
        """
        Calculate deceleration metrics from vehicle trajectory
        """
        # Get initial and final velocities
        initial_speed = trajectory.iloc[0]['vel']
        final_speed = trajectory.iloc[-1]['vel']

        # Speed change (can be negative if vehicle accelerated)
        speed_change = initial_speed - final_speed

        # Time elapsed
        time_elapsed = len(trajectory) * self.sampling_rate
        avg_deceleration = speed_change / time_elapsed if time_elapsed > 0 else 0.0

        # Instantaneous decelerations between consecutive frames
        decelerations = []
        for i in range(len(trajectory) - 1):
            v1 = trajectory.iloc[i]['vel']
            v2 = trajectory.iloc[i + 1]['vel']
            decel = (v1 - v2) / self.sampling_rate
            if decel > 0:  # Only count actual decelerations
                decelerations.append(decel)

        if len(decelerations) == 0:
            max_deceleration = 0.0
            decelerated = False
        else:
            max_deceleration = max(decelerations)
            decelerated = max_deceleration > 0

        # Get initial pair data (first row since grouped)
        pair = initial_pair.iloc[0]

        # Calculate relative velocity
        rel_vel_x = pair['vel_x_veh'] - pair['vel_x_ped']
        rel_vel_y = pair['vel_y_veh'] - pair['vel_y_ped']
        rel_vel = np.sqrt(rel_vel_x**2 + rel_vel_y**2)

        # Create vehicle name
        vehicle_name = self.LABEL_NAMES.get(pair['label_veh'], 'vehicle')

        zone = pair.get('zone', 'unknown')

        return {
            'timestamp': pair['timestamp'],
            'zone': zone,
            'id_obj1': pair['id_ped'],
            'id_obj2': pair['id_veh'],
            'label_obj1': pair['label_ped'],
            'label_obj2': pair['label_veh'],
            'object_pair_labels': f'pedestrian-{vehicle_name}',
            'pos_x_obj1': pair['pos_x_ped'],
            'pos_y_obj1': pair['pos_y_ped'],
            'pos_x_obj2': pair['pos_x_veh'],
            'pos_y_obj2': pair['pos_y_veh'],
            'vel_x_obj1': pair['vel_x_ped'],
            'vel_y_obj1': pair['vel_y_ped'],
            'vel_obj1': pair['vel_ped'],
            'vel_x_obj2': pair['vel_x_veh'],
            'vel_y_obj2': pair['vel_y_veh'],
            'vel_obj2': pair['vel_veh'],
            'yaw_obj1': pair['yaw_ped'],
            'yaw_obj2': pair['yaw_veh'],
            'size_x_obj1': pair['size_x_ped'],
            'size_y_obj1': pair['size_y_ped'],
            'size_x_obj2': pair['size_x_veh'],
            'size_y_obj2': pair['size_y_veh'],
            'rel_dist': pair['distance'],
            'rel_vel': rel_vel,
            'initial_speed': initial_speed,
            'final_speed': final_speed,
            'speed_change': speed_change,
            'avg_deceleration': avg_deceleration,
            'max_deceleration': max_deceleration,
            'observation_time': time_elapsed,
            'num_frames_observed': len(trajectory),
            'alignment': pair['alignment'],
            'decelerated': decelerated
        }

    def _classify_severity(self, max_decel: float) -> str:
        """
        Map max_deceleration to a severity label without filtering.
        """
        # Small tolerance for numerical noise
        if max_decel <= 0.05:
            return 'none'       # no effective deceleration
        if max_decel >= self.DECEL_THRESHOLD_CRITICAL:
            return 'critical'
        if max_decel >= self.DECEL_THRESHOLD_SERIOUS:
            return 'serious'
        if max_decel >= self.DECEL_THRESHOLD_MODERATE:
            return 'moderate'
        return 'low'

    def _empty_result_dataframe(self) -> pd.DataFrame:
        """Return empty DataFrame with correct schema"""
        return pd.DataFrame(columns=[
            'timestamp', 'analytic_id', 'id_obj1', 'id_obj2', 'label_obj1',
            'label_obj2', 'object_pair_labels', 'pos_x_obj1', 'pos_y_obj1',
            'pos_x_obj2', 'pos_y_obj2', 'vel_x_obj1', 'vel_y_obj1', 'vel_obj1',
            'vel_x_obj2', 'vel_y_obj2', 'vel_obj2', 'yaw_obj1', 'yaw_obj2',
            'size_x_obj1', 'size_y_obj1', 'size_x_obj2', 'size_y_obj2',
            'rel_dist', 'rel_vel', 'initial_speed', 'final_speed', 'speed_change',
            'avg_deceleration', 'max_deceleration', 'observation_time',
            'num_frames_observed', 'alignment', 'severity', 'zone', 'decelerated'
        ])


# ======================================================================
# Example usage
# ======================================================================

print("="*80)
print("DECELERATION-BASED PEDESTRIAN-VEHICLE NEAR-MISS DETECTION")
print("="*80)

# Create detector
detector = PedestrianVehicleDecelerationDetector(
    max_pair_distance=3.0,
    min_vehicle_speed=2.0,
    approach_angle_threshold=np.pi/3,
    observation_window=3.0,
    min_deceleration=0.0,  # not used as filter now
    sampling_rate=0.1,
    chunk_size=10000
)

print(f"\nDetector Configuration:")
print(f"  Max pair distance: {detector.max_pair_distance}m")
print(f"  Min vehicle speed: {detector.min_vehicle_speed} m/s")
print(f"  Approach angle: {np.degrees(detector.approach_angle_threshold):.0f}°")
print(f"  Observation window: {detector.observation_window}s")
print(f"  Frames to observe: {detector.frames_to_observe}")

df_1 = df.copy()
print(f"Dataset: {len(df_1)} rows, {df_1['timestamp'].nunique()} timestamps")

# Process
print("\n" + "="*80)
start_time = time.time()
all_pairs, results = detector.process_dataframe(df_1)
elapsed = time.time() - start_time

results['object_pair_ids'] = results['id_obj1'].astype(str) + "_" + results['id_obj2'].astype(str)

print(f"\nProcessing time: {elapsed:.2f} seconds")

print(f"\nTotal candidate pairs (spatial + angle filters, no decel filter): {len(all_pairs)}")
print(f"Total events with decel metrics: {len(results)}")

if len(results) > 0:
    print(f"\n✓ Events with deceleration metrics\n") 
    for idx, event in results.iterrows():
        print("-" * 80)
        print(f"Event #{event['analytic_id']}: {event['severity'].upper()}")
        print("-" * 80)
        print(f"  Timestamp: {event['timestamp']}")
        print(f"  Conflict: {event['object_pair_labels']}")
        print(f"  Initial distance: {event['rel_dist']:.2f} m")
        print(f"  Vehicle alignment: {event['alignment']:.3f}")
        print(f"  Initial speed: {event['initial_speed']:.2f} m/s ({event['initial_speed']*3.6:.1f} km/h)")
        print(f"  Final speed: {event['final_speed']:.2f} m/s ({event['final_speed']*3.6:.1f} km/h)")
        print(f"  Speed reduction: {event['speed_change']:.2f} m/s")
        print(f"  Average deceleration: {event['avg_deceleration']:.2f} m/s²")
        print(f"  Maximum deceleration: {event['max_deceleration']:.2f} m/s²")
        print(f"  Decelerated: {event['decelerated']}")
        print(f"  Observation time: {event['observation_time']:.1f}s ({event['num_frames_observed']} frames)")
        print()
    print("="*80)
    print("SUMMARY")
    print("="*80)
    print(results['severity'].value_counts().to_dict())
    print(f"\nAverage max deceleration: {results['max_deceleration'].mean():.2f} m/s²")
else:
    print("\n✓ No events with valid trajectories")


DECELERATION-BASED PEDESTRIAN-VEHICLE NEAR-MISS DETECTION

Detector Configuration:
  Max pair distance: 3.0m
  Min vehicle speed: 2.0 m/s
  Approach angle: 60°
  Observation window: 3.0s
  Frames to observe: 30
Dataset: 6504009 rows, 787316 timestamps

Processing 6504009 records...
Observation window: 3.0s (30 frames)
Analyzable timestamps: 787286 of 787316
Processing timestamps 7286 of 787286....tion....vehicle 11442064: insufficient trajectory data
✓ Produced 133 events with deceleration metrics

Processing time: 12.82 seconds

Total candidate pairs (spatial + angle filters, no decel filter): 134
Total events with decel metrics: 133

✓ Events with deceleration metrics

--------------------------------------------------------------------------------
Event #1075: LOW
--------------------------------------------------------------------------------
  Timestamp: 2025-06-02 04:40:07.031337975
  Conflict: pedestrian-car
  Initial distance: 2.95 m
  Vehicle alignment: 0.529
  Initial speed: 

In [ ]:
results.drop_duplicates(subset="object_pair_ids")["severity"].value_counts()

severity
low         38
moderate    13
serious     11
none        11
critical     1
Name: count, dtype: int64

In [ ]:
final_df[final_df["object_pair_ids"]== "11666300_11666816"]

,near_miss_timestamp,analytic_id,id_obj1,id_obj2,label_obj1,label_obj2,object_pair_labels,pos_x_obj1,pos_y_obj1,pos_x_obj2,...,traj_min_dist,traj_time_horizon,traj_severity_score,env_min_dist,env_max_overlap_prob,env_time_horizon,env_severity_score,n_models,object_pair_ids,removed_after_cleaning
64,2025-06-02 13:44:32.855501890,1002,11666300,11666816,4,1,1_4,29.1875,17.265625,27.9375,...,0.918029,0.96,0.027315,0.966206,0.353276,1.2,0.32106,3,11666300_11666816,NO


In [ ]:
len(results)

88

In [ ]:
results["object_pair_ids"].nunique()

74

In [ ]:
results[results['severity'] == 'critical']

,timestamp,zone,id_obj1,id_obj2,label_obj1,label_obj2,object_pair_labels,pos_x_obj1,pos_y_obj1,pos_x_obj2,...,speed_change,avg_deceleration,max_deceleration,observation_time,num_frames_observed,alignment,decelerated,analytic_id,severity,object_pair_ids
43,2025-06-02 08:25:58.931521893,1077,11378340,11378291,1,4,pedestrian-car,-1.490234,4.988281,-1.200195,...,2.254113,1.733933,13.379025,1.3,13,0.616363,True,1077,critical,11378340_11378291
